# CLIP detector with LayerNorm tuning

Follows the recipe of GenD ([arXiv 2508.06248](https://arxiv.org/abs/2508.06248)) and [arXiv 2503.19683](https://arxiv.org/abs/2503.19683): a CLIP ViT-L/14 image encoder where only the LayerNorm parameters train, L2-normalized features and a linear head. Batches keep each real photo together with the fakes made from it (`face_decision_lab.pairs`), which GenD reports is essential against shortcut learning. Metric learning from GenD is not implemented yet.

Each protocol in `PROTOCOL_NAMES` trains a fresh model and is scored under `face_decision_lab.protocol`, like the other notebooks.

**Runtime:** Colab Pro, L4 or A100. **Needs** the Colab secret `HF_TOKEN` with write access.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-data"
HF_MODEL = "paulohenriquevn/face-decision-clip-ln"  # private; one folder per protocol
CLIP_MODEL = "openai/clip-vit-large-patch14"
PROTOCOL_NAMES = ("unseen_text2img", "unseen_insight", "subject_holdout", "unseen_inpainting")
BATCH_SIZE = 32
EPOCHS = 1
LEARNING_RATE = 3e-4
TRAIN_CAP = None  # None uses the whole train side of each protocol

### Installation

In [ ]:
%%capture
!pip install -q "face-decision-lab @ {REPO_SOURCE}"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Data

In [ ]:
from face_decision_lab.hub import load_hub_samples

samples, hub_images = load_hub_samples(HF_DATASET)
images = {s.image_path: image for s, image in zip(samples, hub_images, strict = True)}
print(len(samples), "crops")

### Model

In [ ]:
import torch
from torch import nn
from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection

processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL)

def is_layer_norm(name):
    # CLIP vision names: pre_layrnorm, encoder.layers.N.layer_norm1/2, post_layernorm.
    return any(part in name for part in ("layer_norm", "layrnorm", "layernorm"))

class LayerNormTunedCLIP(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL)
        for name, parameter in self.encoder.named_parameters():
            parameter.requires_grad = is_layer_norm(name)
        self.head = nn.Linear(self.encoder.config.projection_dim, 1)

    def forward(self, pixels):
        features = nn.functional.normalize(self.encoder(pixel_values = pixels).image_embeds, dim = -1)
        return self.head(features).squeeze(-1)

probe_model = LayerNormTunedCLIP()
trainable = sum(p.numel() for p in probe_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in probe_model.parameters())
print(f"trainable {trainable:,} of {total:,} ({trainable / total:.3%})")
del probe_model

### Training and scoring

In [ ]:
import json, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from huggingface_hub import HfApi
from tqdm.auto import tqdm
from face_decision_lab.pairs import paired_batches
from face_decision_lab.protocol import SEED, evaluation_report, protocol_split
from face_decision_lab.rows import open_image
from face_decision_lab.splits import capped

pool = ThreadPoolExecutor(max_workers = 8)

def pixels(part):
    batch = list(pool.map(lambda s: open_image(images[s.image_path]), part))
    return processor(images = batch, return_tensors = "pt")["pixel_values"]

def prefetched(batches):
    # Decodes the next batch on CPU threads while the GPU works on the current one.
    with ThreadPoolExecutor(max_workers = 1) as loader:
        future = loader.submit(pixels, batches[0])
        for index in range(len(batches)):
            current = future.result()
            if index + 1 < len(batches):
                future = loader.submit(pixels, batches[index + 1])
            yield batches[index], current

def train_and_score(protocol):
    train, _, test = protocol_split(samples, protocol)
    train = capped(train, TRAIN_CAP, SEED)
    torch.manual_seed(SEED)
    model = LayerNormTunedCLIP().cuda()
    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr = LEARNING_RATE)
    attacks = sum(not s.is_bona_fide for s in train)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight = torch.tensor((len(train) - attacks) / attacks, device = "cuda"))
    epochs = [paired_batches(train, BATCH_SIZE, SEED + epoch) for epoch in range(EPOCHS)]
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = sum(len(b) for b in epochs))
    model.train()
    started = time.time()
    for epoch, batches in enumerate(epochs):
        for step, (batch, x) in enumerate(tqdm(prefetched(batches), total = len(batches), desc = f"{protocol} epoch {epoch}")):
            y = torch.tensor([0.0 if s.is_bona_fide else 1.0 for s in batch], device = "cuda")
            with torch.autocast("cuda", dtype = torch.bfloat16):
                loss = loss_fn(model(x.cuda()).float(), y)
            loss.backward()
            optimizer.step(); optimizer.zero_grad(); scheduler.step()
            if step % 200 == 0:
                print(f"step {step} loss {loss.item():.4f}")
    train_minutes = (time.time() - started) / 60
    model.eval()
    scores = []
    test_batches = [test[i:i + 128] for i in range(0, len(test), 128)]
    with torch.no_grad(), torch.autocast("cuda", dtype = torch.bfloat16):
        for _batch, x in prefetched(test_batches):
            scores += torch.sigmoid(model(x.cuda()).float()).cpu().tolist()
    report = evaluation_report(test, scores)
    report.update(training_rows = len(train), epochs = EPOCHS, train_minutes = round(train_minutes, 1))
    folder = Path(f"/content/clip_ln/{protocol}")
    folder.mkdir(parents = True, exist_ok = True)
    torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, folder / "trained_parameters.pt")
    (folder / "report.json").write_text(json.dumps(report, indent = 2))
    return report, folder

api = HfApi()
api.create_repo(HF_MODEL, private = True, exist_ok = True)
reports = {}
for protocol in PROTOCOL_NAMES:
    report, folder = train_and_score(protocol)
    reports[protocol] = report
    api.upload_folder(folder_path = str(folder), path_in_repo = protocol, repo_id = HF_MODEL, commit_message = f"Add {protocol}")
    print(protocol, json.dumps({k: report[k] for k in ("auc", "eer", "bpcer_at_max_apcer", "training_rows", "train_minutes")}))

### All reports

In [ ]:
print(json.dumps(reports, indent = 2))